### Porting to Google Colab
The following cell enables this notebook to run from Google Colab as well as from your local machine IDE.<br>
You can change `root_directory` and/or `this_notebook_google_path` to point to the directory in your Google account, which contains this notebook, together with the `imgs` sub-directory and the rest of the files.<br>

In [1]:
import sys
import os
try:
    from google.colab import drive as google_drive # type: ignore
except:
    # no Google Colab --> fall back to local machine
    google_drive = None

if google_drive is not None:
    google_drive_directory = os.path.join('/','content','gdrive')
    google_drive.mount(google_drive_directory)
    all_projects_path = os.path.join(google_drive_directory, 'Othercomputers','My Laptop', 'projects')
elif os.name == 'posix': # Ubuntu
    all_projects_path = os.path.join(os.path.expanduser('~'), 'projects')
elif os.name == 'nt': # Windows
    all_projects_path = os.path.join('d:\\', 'projects')
else:
    raise EnvironmentError("Unsupported operating system")
    
project_path = os.path.join(all_projects_path,'RUNI','Thesis')
assert os.path.exists(project_path), f'Project path {project_path} not found!'
# enable import python files from this notebook's path
sys.path.append(project_path)
# enable reading images and data files from this notebook's path
os.chdir(project_path)

datasets_path = os.path.join(project_path, 'datasets')
assert os.path.exists(datasets_path), f'Datasets path {datasets_path} not found!'

output_path = os.path.join(project_path, 'output')
os.makedirs(output_path, exist_ok=True)
assert os.path.exists(output_path), f'Output path {output_path} not found!'
print(f'Current working directory: {os.getcwd()}')
print(f'Datasets path: {datasets_path}')
print(f'Output path: {output_path}')



Current working directory: /home/dror/projects/RUNI/Thesis
Datasets path: /home/dror/projects/RUNI/Thesis/datasets
Output path: /home/dror/projects/RUNI/Thesis/output


In [2]:
from python.hpc import HybridArray

Detecting CUDA version prior to importing numba...
Checking nvcc --version...
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Wed Aug 27 17:34:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.03              Driver Version: 575.64.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        

In [ ]:
import numpy as np
from python.rare_weak_model.rare_weak_model import rare_weak_model
from python.error_controlling_methods.error_controlling_methods import top_k, bonferroni, benjamini_hochberg
from python.hpc import is_use_njit, is_use_gpu

def simulation(shape: tuple, method: str,\
               epsilon: float = 0.01, mu: float = 1.0,\
                **kwargs) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    use_gpu = is_use_gpu(**kwargs)
    use_njit = is_use_njit(**kwargs)
    with (
        HybridArray() as sorted_p_values,
        HybridArray() as counts,
        HybridArray() as num_discoveries
    ):    
        sorted_p_values.realloc(shape=shape, dtype=np.float64, use_gpu=use_gpu)
        n1 = max(np.uint32(1),np.uint32(epsilon*shape[1]))
        rare_weak_model(sorted_p_values_output=sorted_p_values, cumulative_counts_output=counts,\
                        mu=mu, n1=n1, **kwargs)
        if method == 'top_k':
            top_k(sorted_p_values_input=sorted_p_values, num_discoveries_output=num_discoveries, use_njit=use_njit)
        elif method == 'bonferroni':
            bonferroni(sorted_p_values_input=sorted_p_values, num_discoveries_output=num_discoveries, use_njit=use_njit)
        elif method == 'benjamini_hochberg':
            benjamini_hochberg(sorted_p_values_input=sorted_p_values, num_discoveries_output=num_discoveries, use_njit=use_njit)
        else:
            assert False
        ret = (sorted_p_values.numpy(), counts.numpy(), num_discoveries.numpy())
    return ret

def simulation3(shape: tuple, method: str,\
                epsilon: float = 0.01, mu: float = 1.0,\
                    **kwargs) -> None:
    print(f'Running on {method=} {shape=} {epsilon=} {mu=}')
    for gpu,njit in [(False,False), (False,True), (True,False)]:
        p_values, counts, num_discoveries = simulation(shape=shape, use_gpu=gpu, use_njit=njit, method=method, epsilon=epsilon, mu=mu, **kwargs)
        print(f'{gpu=} {njit=} --> p_values.mean={p_values.mean():.2f} counts.mean={counts.mean():.2f} num_discoveries.mean={num_discoveries.mean():.2f}')




In [4]:
shape=(100,1000)
epsilon = 0.1
mu = 1.0
simulation3(shape=shape, method='top_k', epsilon=epsilon, mu=mu)

Running on method='top_k' shape=(100, 1000) epsilon=0.1 mu=1.0
gpu=False njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=500.50
gpu=False njit=True --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=500.50


/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


gpu=True njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=500.50


/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [5]:
simulation3(shape=shape, method='bonferroni', epsilon=epsilon, mu=mu)

Running on method='bonferroni' shape=(100, 1000) epsilon=0.1 mu=1.0
gpu=False njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=525.67
gpu=False njit=True --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=525.67
gpu=True njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=525.67


/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [6]:
simulation3(shape=shape, method='benjamini_hochberg', epsilon=epsilon, mu=mu)

Running on method='benjamini_hochberg' shape=(100, 1000) epsilon=0.1 mu=1.0
gpu=False njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=115.46
gpu=False njit=True --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=115.46
gpu=True njit=False --> p_values.mean=0.47 counts.mean=73.70 num_discoveries.mean=115.46


/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/dror/venv/thesis3.13/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
